# Day S5 — BLAST와 E-value: 데이터베이스 검색과 "우연히 이만큼"의 통계

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/supplement/day-S5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/supplement/day-S5.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 씨앗을 찾고 늘리기

In [ ]:
database = {                      # 가상의 작은 단백질 데이터베이스
    "db1": "MSLLDAKTKEGVVHGLWEQPRS",
    "db2": "MAHHTTPLLQWERRYGGCNV",
    "db3": "GSPKTKQGVVHAAELRKTKE",
}
query = "AKTKEGVVHGA"
K = 3

def score(a, b):
    return 2 if a == b else -1

# 1단계: 데이터베이스의 모든 3글자 단어 위치를 미리 색인해 둔다
index = {}
for name, seq in database.items():
    for pos in range(len(seq) - K + 1):
        index.setdefault(seq[pos:pos + K], []).append((name, pos))

# 2단계: query의 단어가 색인에 있으면 그 자리가 씨앗(seed)
seeds = []
for qpos in range(len(query) - K + 1):
    for name, dpos in index.get(query[qpos:qpos + K], []):
        seeds.append((name, qpos, dpos))
print("씨앗 개수:", len(seeds))

# 3단계: 씨앗을 양쪽으로 gap 없이 늘린다. 점수가 최고점에서 DROP 넘게 떨어지면 멈춘다
DROP = 3
def extend(q, d, qpos, dpos, step):
    best = total = 0
    best_len = length = 0
    while 0 <= qpos < len(q) and 0 <= dpos < len(d):
        total += score(q[qpos], d[dpos])
        length += 1
        if total > best:
            best, best_len = total, length
        if best - total > DROP:
            break
        qpos, dpos = qpos + step, dpos + step
    return best, best_len

hits = {}
for name, qpos, dpos in seeds:
    d = database[name]
    right, right_len = extend(query, d, qpos + K, dpos + K, +1)
    left, left_len = extend(query, d, qpos - 1, dpos - 1, -1)
    total = 2 * K + right + left
    q_start, d_start = qpos - left_len, dpos - left_len
    length = left_len + K + right_len
    hits[(name, q_start, d_start)] = (total, query[q_start:q_start + length], d[d_start:d_start + length])

for (name, q_start, d_start), (total, q_part, d_part) in sorted(hits.items(), key=lambda x: -x[1][0]):
    print(f"{name}  점수 {total:2d}  query {q_start + 1:>2} {q_part}")
    print(f"{'':14s}  db    {d_start + 1:>2} {d_part}")

### 예시 2 — 무작위 서열끼리도 점수는 나온다

In [ ]:
import numpy as np

rng = np.random.default_rng(5)
LETTERS = np.array(list("ACDEFGHIKLMNPQRSTVWY"))

def best_local_score(a, b, match=2, mismatch=-1):
    """gap 없는 최고 국소 점수: 모든 대각선에서 '가장 점수가 높은 연속 구간'을 찾는다."""
    s = np.where(a[:, None] == b[None, :], match, mismatch)
    best = 0
    for k in range(-len(a) + 1, len(b)):
        run = 0
        for value in np.diagonal(s, k):
            run = max(0, run + value)
            best = max(best, run)
    return best

assert best_local_score(np.array(list("AAKTKAA")), np.array(list("CCKTKCC"))) == 6

for length in [50, 100, 200]:
    scores = [best_local_score(rng.choice(LETTERS, length), rng.choice(LETTERS, length))
              for _ in range(300)]
    scores = np.array(scores)
    print(f"길이 {length:3d} × {length:3d}: 무작위 최고점 평균 {scores.mean():.1f}, "
          f"95% 지점 {np.percentile(scores, 95):.0f}, 최댓값 {scores.max()}")

scores = np.array([best_local_score(rng.choice(LETTERS, 100), rng.choice(LETTERS, 100))
                   for _ in range(1000)])
print("\n길이 100 × 100, 1000번 반복한 최고점 분포")
for value in range(scores.min(), scores.max() + 1):
    count = (scores == value).sum()
    print(f"{value:3d} {'#' * (count // 10)} {count}")

### 예시 3 — raw score, bit score, E-value

In [ ]:
import math

LAMBDA, K = 0.267, 0.041      # BLOSUM62 + gap 11/1 조합에서 BLAST가 쓰는 값

def bit_score(raw):
    return (LAMBDA * raw - math.log(K)) / math.log(2)

def e_value(raw, m, n):
    return K * m * n * math.exp(-LAMBDA * raw)

m = 140                        # query 길이 (예: 140 aa 단백질)
print("raw  bit    E(DB 1천만)  E(DB 2억)")
for raw in [40, 60, 80, 120]:
    print(f"{raw:3d}  {bit_score(raw):5.1f}  {e_value(raw, m, 1e7):10.2e}  {e_value(raw, m, 2e8):9.2e}")

print("\nbit score로 다시 계산한 E (raw 60):", f"{m * 1e7 * 2 ** (-bit_score(60)):.2e}")
for raw in [60, 80]:
    e = e_value(raw, m, 1e7)
    print(f"raw {raw}: E = {e:.3f}, P(우연히 1개 이상) = 1 - exp(-E) = {1 - math.exp(-e):.3f}")

## 직접 해보기

4. **E = 0.001이 되려면.** query 140, 데이터베이스 1천만(1e7) 조건에서 E가 0.001 아래로 내려가는 가장 작은 raw score를 `for` 반복문으로 찾습니다.

In [ ]:
for raw in range(40, 200):
    if e_value(raw, 140, 1e7) < 0.001:
        print(raw, round(bit_score(raw), 1))
        break